# ¿Tiene una cola de ley de potencias la población de localidades mexicanas?

Estudio exploratorio con **Principales resultados por localidad (ITER), Censo 2020** de INEGI. Descarga el [ZIP nacional de ITER 2020](https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/iter/iter_00_cpv2020_csv.zip), extrae `conjunto_de_datos/conjunto_de_datos_iter_00CSV20.csv`, renómbralo `iter_00_cpv2020.csv` y consulta el [descriptor de variables](https://www.inegi.org.mx/contenidos/programas/ccpv/2020/doc/fd_iter_cpv2020.pdf). Coloca el CSV en `examples/data/iter_00_cpv2020.csv` y ejecuta el notebook desde `examples/`. No guardes el archivo ni salidas de datos en Git. Instala `pip install -e ".[analysis]"` desde la raíz del proyecto.

La unidad es **localidad censal**, no ciudad ni zona metropolitana. ITER contiene filas de totales y agregados para localidades de una o dos viviendas; se excluyen para evitar contar dos veces. Esto limita la cobertura de localidades pequeñas. La pregunta es si la cola de las localidades con al menos 2 500 habitantes se aproxima mejor a una ley de potencias que a alternativas. El umbral es una decisión previa de este ejercicio, no una estimación automática.


In [ ]:
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path

import pandas as pd

from economista.analysis import fit_power_law

DATA_PATH = Path("data/iter_00_cpv2020.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Descarga y extrae el CSV oficial en {DATA_PATH}")
print("SHA256:", sha256(DATA_PATH.read_bytes()).hexdigest())
print("economista:", version("economista"), "powerlaw:", version("powerlaw"))


## Selección de unidades y limpieza

Los códigos `0000`, `9998` y `9999` designan totales o agregados. Se conservan sólo localidades identificables, con población numérica y positiva. Se reportan los conteos antes y después del filtro.


In [ ]:
raw = pd.read_csv(
    DATA_PATH, encoding="latin-1",
    usecols=lambda name: name.strip().upper() in {"ENTIDAD", "MUN", "LOC", "POBTOT"},
    dtype={"ENTIDAD": "string", "MUN": "string", "LOC": "string"},
    low_memory=False,
)
raw.columns = raw.columns.str.strip().str.upper()
required = {"ENTIDAD", "MUN", "LOC", "POBTOT"}
if missing := required - set(raw.columns):
    raise ValueError(f"Faltan columnas de ITER: {sorted(missing)}")
localities = raw.loc[
    raw["ENTIDAD"].ne("00")
    & raw["MUN"].ne("000")
    & ~raw["LOC"].isin(["0000", "9998", "9999"])
].copy()
localities["value"] = pd.to_numeric(localities["POBTOT"], errors="coerce")
localities = localities.loc[localities["value"].gt(0)].copy()
assert not localities.duplicated(["ENTIDAD", "MUN", "LOC"]).any()
urban = localities.loc[localities["value"].ge(2_500)].copy()
print({"filas_originales": len(raw), "localidades_validas": len(localities),
       "localidades_2500_mas": len(urban)})


## Ajuste y contraste

La población es un conteo entero. Se fija `xmin=2500` para estudiar el tramo definido antes de ver el resultado. La razón de verosimilitud compara modelos sobre la misma cola; su valor p no prueba que la ley de potencias describa bien los datos.


In [ ]:
fit = fit_power_law(urban, discrete=True, xmin=2_500)
print({"alpha": fit.alpha, "xmin": fit.xmin, "ks_distance": fit.ks_distance,
       "sigma": fit.sigma, "n_tail": fit.n_tail})
for alternative in ("lognormal", "exponential"):
    result = fit.compare_to(alternative)
    print(alternative, result.log_likelihood_ratio, result.p_value)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
fit.raw_fit.plot_ccdf(ax=ax, label="Localidades observadas")
fit.raw_fit.power_law.plot_ccdf(ax=ax, label="Ajuste de ley de potencias")
ax.set(
    xlabel="Población por localidad", ylabel="P(X ≥ x)",
    title="Cola de localidades, ITER 2020",
)
ax.legend()
plt.show()


## Interpretación y próximos pasos

Documenta la fecha de descarga, el hash impreso, `n_tail`, las comparaciones y la sensibilidad a otros umbrales antes de divulgar resultados. Una recta visual, un exponente o una comparación favorable no identifican por sí solos un mecanismo de crecimiento urbano. No extiendas esta conclusión a todas las ciudades de México: la unidad es localidad censal y faltan las localidades pequeñas que ITER sólo ofrece como agregados.
